In [2]:
%pip install "dynamiqs>=0.3.0" cmaes scipy
#uncomment to install
#make sure you are using python 3.11 (py -3.11 -m venv .env, then .env\Scripts\activate for Windows)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import dynamiqs as dq
import jax.numpy as jnp
from matplotlib import pyplot as plt

from jax import vmap, jit
from cmaes import SepCMA

from scipy.optimize import curve_fit
from scipy.optimize import least_squares

# dq.set_progress_meter(False)

In [ ]:
#tz = decay of z operator => bit flip
#tx = decay of x operator => phase flip

@jit
def cat_loss_func(x):
  na = 15 # Hilbert space dimension
  a = dq.destroy(na)
  b = dq.destroy(na)

  #x[0] = |g2|, x[1] = phase_g2, x[2] = |ed|, x[3] = phase_ed
  g2 = x[0] * jnp.exp(1j * x[1])
  ed = x[2] * jnp.exp(1j * x[3])
  alpha = jnp.sqrt(x[2] / x[0])
  psi0 = dq.fock(na, 0)
  kappa_2 = 0 #one photon loss rate


  #known hamiltonian for two-qubit dissipation
  H = g2.conj() * (a @ a @ b.dag()) + g2 * (a.dag() @ a.dag() @ b) - ed * b.dag() - ed.conj() * b
  loss_op = jnp.sqrt(kappa_2)*(a @ a - alpha**2 * dq.eye(na)) 
  ts = jnp.linspace(0, 3, 100) #what to put here??

  results = dq.mesolve(
        H,
        [loss_op],
        psi0,
        ts,   
        options=dq.Options(
            progress_meter=False
        )
    )
  
  #minimize deviation from a stable cat qubit
  loss = dq.expect(loss_op.dag() @ loss_op, results.states[-1]).real #is this right??
  print(loss)
  return loss

In [17]:
BATCH_SIZE = 8
N_EPOCHS = 80

batched_cat_loss_func = jit(vmap(cat_loss_func))

mean0 = jnp.array([0.5, 0.5, 0.5, 0.5]) #initial guess
sigma0 = 0.1 #xploration scale

optimizer = SepCMA(
  mean = mean0,
  sigma = sigma0,
  population_size=BATCH_SIZE,
  seed=0
)

# ----------------------------------------
# Logging
# ----------------------------------------
mean_history = []
reward_history = []
reward_std_history = []

# ----------------------------------------
# Training loop
# ----------------------------------------
for epoch in range(N_EPOCHS):
    solutions = []

    # Sample population
    xs = []
    for _ in range(optimizer.population_size):
        xs.append(optimizer.ask())

    xs = jnp.array(xs)
    rewards = batched_cat_loss_func(xs)

    # Format solutions
    solutions = []
    for j in range(len(xs)):
        solutions.append((xs[j], rewards[j]))

    optimizer.tell(solutions)

    # Log
    mean_history.append(optimizer.mean.copy())
    reward_history.append(jnp.mean(rewards))
    reward_std_history.append(jnp.std(rewards))

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | mean={optimizer.mean} | avg reward={jnp.mean(rewards):.4f}")


Traced<float32[]>with<DynamicJaxprTrace>
Epoch   0 | mean=[0.6431867 0.4995537 0.5803265 0.6314609] | avg reward=0.0000
Epoch  10 | mean=[0.48271075 0.47810015 0.31386292 0.85517097] | avg reward=0.0000
Epoch  20 | mean=[0.52457315 0.24083331 0.24713667 0.7767941 ] | avg reward=0.0000
Epoch  30 | mean=[0.47484723 0.30740258 0.4819695  0.77931607] | avg reward=0.0000
Epoch  40 | mean=[0.49229395 0.32025483 0.45368123 0.7409467 ] | avg reward=0.0000
Epoch  50 | mean=[0.49543306 0.3029623  0.44678637 0.7403833 ] | avg reward=0.0000
Epoch  60 | mean=[0.49639663 0.30976036 0.4368571  0.74088615] | avg reward=0.0000
Epoch  70 | mean=[0.45095658 0.30003166 0.4090299  0.7478532 ] | avg reward=0.0000
